# Medicare coverage policies (LCDs) download
Run the cells in order (about 2 minutes). The last cell downloads a small zip to your computer.

In [ ]:
# Step 1: download and unpack the current LCD export (about 40 MB)
import io
import shutil
import zipfile
from pathlib import Path

import pandas as pd
import requests

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/130.0 Safari/537.36"
    )
}
URL = "https://downloads.cms.gov/medicare-coverage-database/downloads/exports/current_lcd.zip"
WORK = Path("/content/lcd_raw")
OUT = Path("/content/coverage")
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir(parents=True)
OUT.mkdir(parents=True, exist_ok=True)

r = requests.get(URL, headers=HEADERS, timeout=600)
print("HTTP", r.status_code, f"{len(r.content) / 1e6:.0f} MB")
r.raise_for_status()


def unpack(data, folder):
    """Extract a zip, and any zips inside it."""
    with zipfile.ZipFile(io.BytesIO(data)) as z:
        for name in z.namelist():
            if name.lower().endswith(".zip"):
                unpack(z.read(name), folder / Path(name).stem)
            elif not name.endswith("/"):
                target = folder / name
                target.parent.mkdir(parents=True, exist_ok=True)
                target.write_bytes(z.read(name))


unpack(r.content, WORK)
csvs = sorted(WORK.rglob("*.csv"))
print(f"{len(csvs)} CSV files, for example:")
for p in csvs[:15]:
    print("  ", p.relative_to(WORK))

In [ ]:
# Step 2: find the main LCD table and the policies for our 5 procedures
def read_csv(path):
    return pd.read_csv(path, dtype=str, encoding="latin-1", keep_default_na=False, low_memory=False)


main_table = None
for p in csvs:
    header = pd.read_csv(p, nrows=0, encoding="latin-1").columns.str.lower().tolist()
    if "lcd_id" in header and "title" in header and len(header) > 10:
        main_table = p
        break
print("Main LCD table:", main_table.relative_to(WORK))

lcd = read_csv(main_table)
lcd.columns = lcd.columns.str.lower()
print(f"{len(lcd):,} LCD rows, columns:")
print(", ".join(lcd.columns))

TOPICS = {
    "blepharoplasty": ["blepharoplasty", "eyelid"],
    "epidural_steroid_injection": ["epidural"],
    "facet_joint": ["facet joint"],
    "sleep_study": ["polysomnography", "sleep"],
    "cataract": ["cataract"],
}

rows = []
for topic, words in TOPICS.items():
    hit = lcd[lcd["title"].str.lower().str.contains("|".join(words))]
    print(f"\n{topic}: {len(hit)} LCDs")
    for t in hit["title"].drop_duplicates().head(8):
        print("   -", t)
    rows.append(hit.assign(topic=topic))

selected = pd.concat(rows, ignore_index=True)
selected.to_csv(OUT / "lcd_selected.csv", index=False)
print(f"\nSaved {len(selected)} LCD rows for {selected['topic'].nunique()} topics")

In [ ]:
# Step 3: zip and download to your computer
from google.colab import files

shutil.make_archive("/content/coverage", "zip", OUT)
files.download("/content/coverage.zip")
print("Download started: coverage.zip")